In [14]:
import re

import pandas as pd
import requests

URL = "https://courselistings.wpi.edu/assets/prod-data.json"
# splits "CS 1101 - Introduction to Program Design" into code + title
TITLE_RE = re.compile(r"^(?P<code>[A-Za-z]+ ?\d+[A-Za-z]*) - (?P<title>.*)$")
HTML_TAG_RE = re.compile(r"<[^>]+>")

In [15]:
# pull raw course section entries (one per offered section, not per unique course)
entries = requests.get(URL, timeout=30).json()["Report_Entry"]
print(f"Pulled {len(entries)} course section entries from WPI course listings")

Pulled 3768 course section entries from WPI course listings


In [16]:
rows = []
for e in entries:
    m = TITLE_RE.match(e["Course_Title"])
    rows.append(
        {
            "course_code": m["code"],
            "title": m["title"],
            "description": HTML_TAG_RE.sub("", e["Course_Description"]),
            "subject": e["Subject"],
            "credits": float(e["Credits"]),
            "academic_level": e["Academic_Level"],
            "department": e["Course_Section_Owner"],
            "start_date": e["Course_Section_Start_Date"],
        }
    )

courses_df = pd.DataFrame(rows).sort_values("start_date")
# collapse sections down to one row per course, keeping the most recent offering
courses_df = courses_df.drop_duplicates("course_code", keep="last").drop(columns="start_date")
courses_df = courses_df.set_index("course_code")

print(len(courses_df), "unique courses")
courses_df.head()

1282 unique courses


,title,description,subject,credits,academic_level,department
course_code,,,,,,
AB 1531,Elementary Arabic I,Cat. IAn intensive course to introduce the Ara...,Arabic,3.0,Undergraduate,Humanities and Arts Department
ME 500,Applied Analytical Methods in Engineering,The emphasis of this course is on the modeling...,Mechanical Engineering,3.0,Graduate,Mechanical and Materials Engineering Department
CS 557,Software Security Design And Analysis,Software is responsible for enforcing many cen...,Computer Science,3.0,Graduate,Computer Science Department
CS 555,Responsible Artificial Intelligence,CS 555 / DS 555: Responsible Artificial Intell...,Computer Science; Data Science,3.0,Graduate,Computer Science Department
CS 554,Natural Language Processing,CS 554 / DS 554: Natural Language Processing (...,Computer Science; Data Science,3.0,Graduate,Computer Science Department


In [17]:
# map each department to the list of course codes it owns
courses_by_dept = courses_df.groupby("department").apply(
    lambda g: list(g.index), include_groups=False
)
departments_df = pd.DataFrame({"course_codes": courses_by_dept})

print(len(departments_df), "departments")
departments_df.head()

30 departments


,course_codes
department,
Aerospace Engineering Department,"[AE 601, AE 5233, AE 4220, AE 4210, AE 3310, A..."
Air Force Aerospace Studies (AFROTC) Department,"[AS 2001, AS 1001, AS 4001, AS 3001, AS 1002, ..."
Bioinformatics and Computational Biology Program,"[CS 583, BCB 503, BCB 555, BB 581, BCB 501, BC..."
Biology and Biotechnology Department,"[BB 575, BB 552, BB 504, BB 560, BB 2815, BB 3..."
Biomedical Engineering Department,"[ME 4814, ECE 4023, BME 4023, BME 4813, BME 59..."


In [18]:
courses_df.to_csv("courses.csv")
departments_df.to_csv("departments.csv")